In [1]:
import os
import numpy as np
import pandas as pd
import scipy.integrate
from scipy.interpolate import interp1d

# Fix for scipy update breaking PyMieScatt
if not hasattr(scipy.integrate, 'trapz'):
    scipy.integrate.trapz = scipy.integrate.trapezoid

import PyMieScatt as ps
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt

In [ ]:
def load_material_data(filename, target_wavelengths_nm):
    """
    Loads empirical (n, k) data from refractiveindex.info CSVs.
    Interpolates the data to match the simulation's wavelength grid.
    """
    # If a file is missing, fallback to a dummy dielectric to prevent crashes
    if not os.path.exists(filename):
        print(f"WARNING: {filename} not found. Using default n=1.5, k=0")
        return np.full_like(target_wavelengths_nm, 1.5 + 0.0j, dtype=np.complex128)

    df = pd.read_csv(filename)
    
    # refractiveindex.info outputs wavelength in micrometers (um). 
    # PyMieScatt requires nanometers (nm).
    csv_wl_nm = df['wl'].values * 1000.0
    n_values = df['n'].values
    
    # Some materials (pure dielectrics) might not have a 'k' column in the CSV
    k_values = df['k'].values if 'k' in df.columns else np.zeros_like(n_values)

    # Create interpolation functions
    # fill_value='extrapolate' handles target wavelengths slightly outside the CSV bounds
    n_interp = interp1d(csv_wl_nm, n_values, kind='linear', fill_value='extrapolate')
    k_interp = interp1d(csv_wl_nm, k_values, kind='linear', fill_value='extrapolate')

    # Generate the complex array for the specific target wavelengths
    n_target = n_interp(target_wavelengths_nm)
    k_target = k_interp(target_wavelengths_nm)
    
    return n_target + 1j * k_target

In [ ]:
class NanoAntennaEnv(gym.Env):
    """Custom Environment for Inverse Design with Empirical Material Dispersion"""
    metadata = {'render.modes': ['console']}

    def __init__(self):
        super(NanoAntennaEnv, self).__init__()
        
        # Wavelength grid (300nm to 1000nm)
        self.wavelengths = np.linspace(300, 1000, 50)
        self.target_spectrum = None
        self.current_step = 0
        self.max_steps = 10 
        
        # Load empirical data arrays for all 7 materials
        self.materials = {
            0: load_material_data('Ag.csv', self.wavelengths),
            1: load_material_data('Au.csv', self.wavelengths),
            2: load_material_data('Cu.csv', self.wavelengths),
            3: load_material_data('Al.csv', self.wavelengths),
            4: load_material_data('Si.csv', self.wavelengths),
            5: load_material_data('Ge.csv', self.wavelengths),
            6: load_material_data('GaAs.csv', self.wavelengths)
        }

        # ---------------- ACTION SPACE ----------------
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(4,), dtype=np.float32
        )

        # Updated bounds for 7 discrete material choices (0 through 6)
        self.bounds = {
            "r_core": (10.0, 200.0),
            "t_shell": (5.0, 100.0),
            "mat": (0.0, 6.999) 
        }

        # ---------------- OBSERVATION SPACE ----------------
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(100,), dtype=np.float32
        )

    def _scale_action(self, norm_val, min_val, max_val):
        return min_val + (max_val - min_val) * ((norm_val + 1.0) / 2.0)

    def _get_spectrum(self, action):
        r_core = self._scale_action(action[0], self.bounds["r_core"][0], self.bounds["r_core"][1])
        t_shell = self._scale_action(action[1], self.bounds["t_shell"][0], self.bounds["t_shell"][1])
        r_total = r_core + t_shell
        
        mat_core_idx = int(self._scale_action(action[2], self.bounds["mat"][0], self.bounds["mat"][1]))
        mat_shell_idx = int(self._scale_action(action[3], self.bounds["mat"][0], self.bounds["mat"][1]))
        
        # Retrieve the complex refractive index ARRAYS for the selected materials
        m_core_array = self.materials[mat_core_idx]
        m_shell_array = self.materials[mat_shell_idx]
        
        spectrum = []
        # Calculate Mie efficiencies using wavelength-dependent (n, k)
        for i, wl in enumerate(self.wavelengths):
            m_core = m_core_array[i]
            m_shell = m_shell_array[i]
            
            qext, _, _, _, _, _, _ = ps.MieQCoreShell(
                m_core, m_shell, wl, r_core*2, r_total*2, 1.0
            )
            spectrum.append(qext)
            
        return np.array(spectrum, dtype=np.float32)

    def step(self, action):
        self.current_step += 1
        current_spectrum = self._get_spectrum(action)
        
        dot_product = np.dot(current_spectrum, self.target_spectrum)
        norm_a = np.linalg.norm(current_spectrum)
        norm_b = np.linalg.norm(self.target_spectrum)
        
        if norm_a == 0 or norm_b == 0:
            cosine_sim = -1.0 
        else:
            cosine_sim = dot_product / (norm_a * norm_b)
            
        reward = float(cosine_sim) * 10.0 if cosine_sim > 0.95 else float(cosine_sim)
        observation = np.concatenate([current_spectrum, self.target_spectrum])
        
        terminated = cosine_sim > 0.98  
        truncated = self.current_step >= self.max_steps
        
        return observation, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        
        # Reverting to random targets for generalized training
        random_action = self.action_space.sample()
        self.target_spectrum = self._get_spectrum(random_action)
        
        blank_spectrum = np.zeros_like(self.target_spectrum)
        observation = np.concatenate([blank_spectrum, self.target_spectrum])
        
        return observation, {}

In [ ]:
if __name__ == "__main__":
    log_dir = "./nanoantenna_logs/"
    os.makedirs(log_dir, exist_ok=True)

    env = NanoAntennaEnv()
    env = Monitor(env, log_dir) 

    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003)

    model.learn(total_timesteps=100000) 

    x, y = ts2xy(load_results(log_dir), 'timesteps')
    plt.figure(figsize=(10, 5))
    plt.plot(x, y, color='blue', alpha=0.3)
    
    if len(x) > 0:
        window = min(100, len(x))
        smoothed_y = np.convolve(y, np.ones(window)/window, mode='valid')
        plt.plot(x[len(x)-len(smoothed_y):], smoothed_y, color='red', linewidth=2, label="Trendline")

    plt.xlabel('Timesteps')
    plt.ylabel('Episode Reward (Cosine Similarity)')
    plt.title('RL Curve: Empirical Material Dispersion')
    plt.legend()
    plt.grid(True)
    plt.show()